In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# LiF — powder X-ray CW — single wavelength

Verifies the baseline LiF Cu Kα₁ pattern with a pseudo-Voigt profile
and no polarization or absorption correction.

**Refinement:** the overall scale only; all other parameters are
taken from the FullProf reference.

In [2]:
import easydiffraction as edi
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = edi.Project()

## Define the structure

In [4]:
structure = StructureFactory.from_scratch(name='lif')

structure.space_group.name_h_m = 'F m -3 m'  # FullProf Space group symbol
structure.cell.length_a = 4.026700  # FullProf a

structure.atom_sites.create(
    id='Li1',  # FullProf Atom
    type_symbol='Li',  # FullProf Typ
    fract_x=0.0,  # FullProf X
    fract_y=0.0,  # FullProf Y
    fract_z=0.0,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=1.20000,  # FullProf Biso
)
structure.atom_sites.create(
    id='F1',  # FullProf Atom
    type_symbol='F',  # FullProf Typ
    fract_x=0.5,  # FullProf X
    fract_y=0.5,  # FullProf Y
    fract_z=0.5,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.80000,  # FullProf Biso
)

project.structures.add(structure)

## Load the FullProf reference

In [5]:
FULLPROF_PROJECT_DIR = 'pd-xray-cwl_lif'
FULLPROF_PRF_FILE = 'lif_single_unpolarized.prf'
FULLPROF_SUM_FILE = 'lif_single_unpolarized.sum'
FULLPROF_BAC_FILE = 'lif_single_unpolarized.bac'
FULLPROF_LABEL = verify.fullprof_label(FULLPROF_PROJECT_DIR, FULLPROF_SUM_FILE)

FULLPROF_ZERO = 0.0  # FullProf Zero
FULLPROF_SCALE = 0.01  # FullProf Scale
FULLPROF_WAVELENGTH = 1.540560  # FullProf Lambda1
FULLPROF_U = 0.048457  # FullProf U
FULLPROF_V = -0.083053  # FullProf V
FULLPROF_W = 0.040000  # FullProf W
FULLPROF_X = 0.0  # FullProf X
FULLPROF_Y = 0.049268  # FullProf Y
FULLPROF_WDT = 48.0  # FullProf Wdt

x, calc_fullprof = verify.load_fullprof_calc_profile(
    FULLPROF_PROJECT_DIR,
    FULLPROF_PRF_FILE,
    FULLPROF_BAC_FILE,
    FULLPROF_ZERO,
)

## Create the experiment

In [6]:
experiment = ExperimentFactory.from_scratch(
    name='lif',
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='xray',
    scattering_type='bragg',
)
verify.set_reference_as_measured(experiment, x, calc_fullprof)

experiment.linked_structures.create(structure_id='lif', scale=FULLPROF_SCALE)

experiment.instrument.setup_wavelength = FULLPROF_WAVELENGTH
experiment.instrument.calib_twotheta_offset = FULLPROF_ZERO

experiment.peak.type = 'pseudo-voigt'
experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y

experiment.peak.cutoff_fwhm = FULLPROF_WDT

project.experiments.add(experiment)

Peak profile type for experiment 'lif' changed to


pseudo-voigt


## edi-cryspy VS FullProf

In [7]:
experiment.calculator.type = 'cryspy'

project.analysis.calculate()
calc_ed_cryspy = experiment.data.intensity_calc
LABEL_ED_CRYSPY = verify.engine_label('cryspy')

project.display.pattern_comparison(
    'lif',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy,
    reference_label=FULLPROF_LABEL,
    candidate_label=LABEL_ED_CRYSPY,
)

Calculator for experiment 'lif' already set to


cryspy


## edi-crysfml VS FullProf

In [8]:
experiment.calculator.type = 'crysfml'

project.analysis.calculate()
calc_ed_crysfml = experiment.data.intensity_calc
LABEL_ED_CRYSFML = verify.engine_label('crysfml')

project.display.pattern_comparison(
    'lif',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml,
    reference_label=FULLPROF_LABEL,
    candidate_label=LABEL_ED_CRYSFML,
)

Calculator for experiment 'lif' changed to


crysfml


⚠️ [CrysfmlCalculator] peak.cutoff_fwhm is not applied by the CrysFML backend (it uses a fixed CFL peak window); the value is     
   ignored.                                                                                                                       


## Fit edi-crysfml to FullProf

In [9]:
experiment.linked_structures['lif'].scale.free = True

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_crysfml_refined = experiment.data.intensity_calc
LABEL_ED_CRYSFML_REFINED = verify.engine_label('crysfml', note='refined')

project.display.pattern_comparison(
    'lif',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml_refined,
    reference_label=FULLPROF_LABEL,
    candidate_label=LABEL_ED_CRYSFML_REFINED,
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'lif' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),1.39
4,🔁 Iterations,5
5,📏 Goodness-of-fit (reduced χ²),0.00
6,"📏 R-factor (Rf, %)",0.22
7,"📏 R-factor squared (Rf², %)",0.05
8,"📏 Weighted R-factor (wR, %)",0.05


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lif,linked_structure,lif,scale,,0.0100,0.0100,0.0000,0.03 % ↓


⚠️ [CrysfmlCalculator] peak.cutoff_fwhm is not applied by the CrysFML backend (it uses a fixed CFL peak window); the value is     
   ignored.                                                                                                                       


## Agreement check

In [10]:
verify.assert_patterns_agree(
    [
        (f'{LABEL_ED_CRYSPY} vs {FULLPROF_LABEL}', calc_fullprof, calc_ed_cryspy),
        (
            f'{LABEL_ED_CRYSFML_REFINED} vs {FULLPROF_LABEL}',
            calc_fullprof,
            calc_ed_crysfml_refined,
        ),
    ],
)

,Comparison,Metric,Expected,Actual,OK
1,edi 0.18.0+dev52 (cryspy 0.12.1) vs FullProf 7.95,Profile diff (%),< 2.5,0.23,✅
2,,Max deviation (%),< 6,0.19,✅
3,,Area ratio,0.99 to 1.01,1.0003,✅
4,,Shape correlation,> 0.999,1.0000,✅
5,"edi 0.18.0+dev52 (crysfml 0.7.0, refined) vs FullProf 7.95",Profile diff (%),< 2.5,0.05,✅
6,,Max deviation (%),< 6,0.01,✅
7,,Area ratio,0.99 to 1.01,0.9981,✅
8,,Shape correlation,> 0.999,1.0000,✅


True